In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def normalize_key(value):
    value = str(value).upper()
    return re.sub(r"[^A-Z0-9]", "", unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode())

points_file = PROCESSED_DIR / "data_points.csv"
population_file = PROCESSED_DIR / "population.csv"
if not points_file.exists() or not population_file.exists():
    raise FileNotFoundError("Executez 02_nettoyage.ipynb avant 04_indicateurs.ipynb")

data_points = pd.read_csv(points_file)
population = pd.read_csv(population_file)

data_points["commune_key"] = data_points["commune_nom_bdd"].fillna("").map(normalize_key)
commune_metrics = (
    data_points.groupby(["region_nom_bdd", "commune_nom_bdd"], dropna=False)
    .agg(
        points=("service", "size"),
        mobile_money=("service", lambda values: int((values == "Mobile Money").sum())),
        infrastructures=("service", lambda values: int((values != "Mobile Money").sum())),
        services=("service", "nunique"),
    )
    .reset_index()
)
commune_metrics["commune_key"] = commune_metrics["commune_nom_bdd"].fillna("").map(normalize_key)
commune_metrics = commune_metrics.merge(population, on="commune_key", how="left")
commune_metrics["pop"] = commune_metrics["pop"].fillna(0)
commune_metrics = commune_metrics[commune_metrics["pop"] > 0].copy()
commune_metrics["points_per_10k"] = commune_metrics["points"] / commune_metrics["pop"] * 10000
commune_metrics["mobile_money_per_10k"] = commune_metrics["mobile_money"] / commune_metrics["pop"] * 10000
commune_metrics["infrastructures_per_100k"] = commune_metrics["infrastructures"] / commune_metrics["pop"] * 100000

region_metrics = commune_metrics.groupby("region_nom_bdd", as_index=False).agg(
    population=("pop", "sum"), points=("points", "sum"), mobile_money=("mobile_money", "sum"), infrastructures=("infrastructures", "sum")
)
region_metrics["points_per_10k"] = region_metrics["points"] / region_metrics["population"] * 10000
region_metrics["mobile_money_per_10k"] = region_metrics["mobile_money"] / region_metrics["population"] * 10000
region_metrics.to_csv(OUTPUT_DIR / "indicateurs_regions.csv", index=False)
commune_metrics.to_csv(OUTPUT_DIR / "indicateurs_communes.csv", index=False)

display(region_metrics.sort_values("points_per_10k", ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=region_metrics.sort_values("points_per_10k"), y="region_nom_bdd", x="points_per_10k", ax=axes[0], color="#26a69a")
axes[0].set_title("Points recensés pour 10 000 habitants")
axes[0].set_xlabel("Points / 10 000 habitants")
axes[0].set_ylabel("")
sns.scatterplot(data=region_metrics, x="population", y="points_per_10k", size="points", hue="mobile_money_per_10k", palette="viridis", ax=axes[1])
axes[1].set_title("Population et intensité de desserte")
axes[1].set_xlabel("Population RGPH-5")
axes[1].set_ylabel("Points / 10 000 habitants")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "indicateurs_regionaux.png", dpi=160)
plt.show()

commune_metrics["priorite"] = ((1 - commune_metrics["mobile_money_per_10k"].rank(pct=True)) * 50 + commune_metrics["pop"].rank(pct=True) * 25 + commune_metrics["infrastructures"].eq(0).astype(int) * 25).round().astype(int)
priorites = commune_metrics.sort_values(["priorite", "pop"], ascending=False).head(20)
display(priorites[["commune_nom_bdd", "region_nom_bdd", "pop", "mobile_money_per_10k", "infrastructures", "priorite"]])
priorites.to_csv(OUTPUT_DIR / "communes_prioritaires.csv", index=False)